[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees/corrections/seance4_correction.ipynb)

# Séance 2.4 — Visualiser et conclure — étude de cas

**Correction** · durée : 2h — 1h de technique en alternance, 1h d'étude de cas en binôme

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- choisir le bon graphique selon la question posée
- produire une courbe, un histogramme, un diagramme en barres et un nuage de points
- rendre un graphique lisible : titre, axes, unités
- repérer ce qu'un graphique cache autant que ce qu'il montre
- conclure une analyse par des recommandations chiffrées

## Pourquoi faire un graphique

Voici le chiffre d'affaires mensuel, sous forme de tableau :

| mois | CA | mois | CA |
|---|---|---|---|
| 2010-12 | 57 705 | 2011-06 | 75 590 |
| 2011-01 | 78 452 | 2011-07 | 107 164 |
| 2011-02 | 52 115 | 2011-08 | 85 469 |
| 2011-03 | 80 451 | 2011-09 | 133 236 |
| 2011-04 | 60 493 | 2011-10 | 170 010 |
| 2011-05 | 80 393 | 2011-11 | 133 937 |

Vous l'avez lu. Avez-vous **vu** quelque chose ?

Maintenant regardez la même chose en courbe (dans deux cellules). La tendance
saute aux yeux en une seconde.

> **Un graphique ne décore pas un rapport : il fait voir ce qu'un tableau
> cache.** Corollaire souvent oublié — si un tableau de trois lignes suffit,
> ne faites pas de graphique.

## Choisir le bon graphique

| Votre question | Le graphique |
|---|---|
| Comment ça évolue **dans le temps** ? | une **courbe** |
| Qui est le plus gros ? Comment ça se **compare** ? | des **barres** |
| Comment les valeurs sont-elles **réparties** ? | un **histogramme** |
| Y a-t-il un **lien** entre deux grandeurs ? | un **nuage de points** |

Quatre questions, quatre graphiques. C'est presque tout ce dont vous aurez
besoin.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")
clients = pd.read_csv(BASE + "clients.csv")
produits = pd.read_csv(BASE + "produits.csv")

ventes["ca"] = ventes["qte"] * ventes["prix"]     ## le CA de chaque ligne
ventes["date"] = pd.to_datetime(ventes["date"])   ## du texte vers des dates

# Deux jointures enchainees : ventes + clients, puis + produits
complet = ventes.merge(clients, on="client_id").merge(produits, on="prod_id")
print(complet.shape)   ## 45 123 lignes : aucune perdue en chemin

## 1. La courbe — l'évolution dans le temps

In [ ]:
# to_period("M") regroupe toutes les dates d'un meme mois
ca_mois = ventes.groupby(ventes["date"].dt.to_period("M"))["ca"].sum()
ca_mois.index = ca_mois.index.astype(str)   ## en texte : matplotlib prefere

ca_mois.plot(kind="line", marker="o", figsize=(7, 4))   ## une evolution
plt.title("Chiffre d'affaires mensuel")   ## sans titre, ce n'est pas un livrable
plt.ylabel("CA (euros)")                  ## la grandeur ET son unite
plt.xticks(rotation=45)                   ## des dates inclinees se lisent
plt.show()

Une montée régulière jusqu'à un pic en octobre, puis une chute brutale en
décembre.

**Que concluez-vous ?** Prenez trente secondes avant de continuer.

In [ ]:
# Verifions quelque chose avant de conclure
decembre = ventes.query("date >= '2011-12-01'")

print("derniere date du fichier :", ventes["date"].max().date())   ## le 9 !
print("jours de decembre 2011 presents :", decembre["date"].dt.day.nunique())

> ⚠️ **Il n'y a pas eu d'effondrement en décembre.** Le fichier s'arrête au
> **9 décembre**. On compare 8 jours de vente à des mois complets de 30 jours.
>
> Le graphique ne mentait pas. C'est la lecture qui était fausse.

C'est **l'erreur d'analyse la plus fréquente en entreprise**, et l'une des
plus coûteuses. Avant d'interpréter une évolution, vérifiez toujours que
**toutes les périodes sont comparables**.

Le vrai pic, lui, est bien réel : octobre. Pour un grossiste, c'est logique —
les détaillants se réapprovisionnent **avant** Noël, pas pendant.

---

### ✏️ À vous 1a — La courbe, avec titre et unité

> **Votre mission :**
> - Tracer le **nombre de commandes distinctes** par mois sous forme de courbe.
> - Titre et unité obligatoires : un graphique sans légende n'est pas un graphique, c'est un dessin.
> - Incliner les étiquettes à 45° et appeler `tight_layout()` : sur un petit écran, sans ça les dates se chevauchent ou sont coupées.
> - Mettre le mois qui compte le plus de commandes dans `mois_cmd`, et le nombre de mois du fichier dans `nb_mois`.

In [ ]:
# nunique() et non count() : une commande de 30 articles reste UNE commande
nb_cmd = complet.groupby(complet["date"].dt.to_period("M"))["cmd_id"].nunique()
nb_cmd.index = nb_cmd.index.astype(str)   ## en texte pour l'affichage

nb_cmd.plot(kind="line", marker="o", figsize=(7, 4))   ## une evolution
plt.title("Nombre de commandes par mois")
plt.ylabel("commandes")     ## la grandeur, pas "valeurs"
plt.xticks(rotation=45)     ## des dates inclinees ne se chevauchent pas
plt.tight_layout()          ## rien ne sera coupe au bord
plt.show()

# Novembre est le mois qui compte le plus de commandes. Le mois le plus
# fort en EUROS est un autre : vous le trouverez en partie 2.
mois_cmd = nb_cmd.idxmax()
nb_mois = len(nb_cmd)   ## 13 : decembre 2010 ET decembre 2011
print(mois_cmd, "|", nb_mois, "mois")

In [ ]:
verifier("1a - mois record en commandes", mois_cmd == "2011-11",
         "nunique() compte les commandes distinctes, count() compterait les lignes")
verifier("1b - nombre de mois", nb_mois == 13,
         "decembre 2010 et decembre 2011 comptent tous les deux")

---

### ✏️ À vous 1b — Deux marchés sur la même figure

> **Votre mission :**
> - Comparer l'évolution mensuelle de la France et de l'Allemagne **sur un seul graphique**.
> - Mettre le meilleur mois français dans `mois_fr`.
> - Deux courbes sur une figure se comparent ; deux figures côte à côte, non.

In [ ]:
deux = complet.query("pays in ['France', 'Allemagne']")   ## deux marches

# unstack() met les pays en colonnes : une colonne = une courbe
par_mois = deux.groupby([deux["date"].dt.to_period("M"), "pays"])["ca"].sum().unstack()
par_mois.index = par_mois.index.astype(str)

par_mois.plot(kind="line", marker="o", figsize=(7, 4))   ## deux courbes d'un coup
plt.title("France et Allemagne, mois par mois")
plt.ylabel("CA (euros)")
plt.show()

mois_fr = par_mois["France"].idxmax()   ## le mois, pas le montant
print(mois_fr)

In [ ]:
verifier("1c - meilleur mois francais", mois_fr == "2011-10",
         "les deux pays sont France et Allemagne")

## 2. Les barres — comparer

**Toujours trier avant de tracer.** Un diagramme en barres non trié est
illisible.

In [ ]:
# nlargest(8) : le top 8. sort_values() : matplotlib dessine de bas en haut
ca_pays = complet.groupby("pays")["ca"].sum().nlargest(8).sort_values()

ca_pays.plot(kind="barh", figsize=(7, 4))   ## barh : les noms se lisent
plt.title("Chiffre d'affaires par pays (top 8)")
plt.xlabel("CA (euros)")
plt.show()

> 💡 **`barh` plutôt que `bar`.** En barres horizontales, les noms se lisent
> sans se chevaucher et sans rotation. Sur un écran étroit, c'est décisif.
>
> Et `sort_values()` **sans** `ascending=False` : matplotlib dessine de bas en
> haut, donc trier en ordre croissant met le plus grand tout en haut.

---

### ✏️ À vous 2a — Le rythme de la semaine

> **Votre mission :**
> - Calculer le CA par **jour de la semaine** dans `ca_jour`, trié du plus petit au plus grand.
> - Le tracer en barres horizontales, avec titre et unité.
> - Mettre le jour le plus fort dans `jour_top`.

In [ ]:
# .dt.day_name() donne le nom du jour de chaque date
ca_jour = complet.groupby(complet["date"].dt.day_name())["ca"].sum().sort_values()

# barh plutot que bar : les noms de jours tiennent a l'horizontale
ca_jour.plot(kind="barh", figsize=(7, 4))   ## 7 pouces sur 4 : lisible partout
plt.title("Chiffre d'affaires par jour de la semaine")
plt.xlabel("CA (euros)")
plt.show()

# Six barres seulement : le samedi n'existe pas dans ce fichier
jour_top = ca_jour.index[-1]
print(jour_top)

In [ ]:
verifier("2a - jour le plus fort", jour_top == "Thursday",
         "sort_values() trie ; apres un tri croissant le plus grand est en position -1")

---

### ✏️ À vous 2b — Compter n'est pas sommer

> **Votre mission :**
> - Combien de **références différentes** chaque catégorie contient-elle ? → `nb_ref`, trié, en barres horizontales.
> - Mettre la catégorie la plus fournie dans `cat_ref`.
> - Attention : on compte des produits, on n'additionne pas des euros. Ce n'est pas le même graphique ni la même conclusion.

In [ ]:
# size() compte les lignes de chaque groupe ; sum() additionnerait des euros
nb_ref = produits.groupby("categorie").size().sort_values()

nb_ref.plot(kind="barh", figsize=(7, 4))
plt.title("Nombre de references par categorie")
plt.xlabel("references")
plt.show()

# La categorie la plus FOURNIE n'est pas forcement celle qui rapporte
# le plus : on le verra en partie 2.
cat_ref = nb_ref.index[-1]
print(cat_ref)

In [ ]:
verifier("2b - categorie la plus fournie", cat_ref == "deco",
         "size() compte les lignes, sum() additionnerait des valeurs")

## 3. L'histogramme — la répartition

In [ ]:
ventes["prix"].plot(kind="hist", bins=50, figsize=(7, 4))   ## 50 classes
plt.title("Repartition des prix unitaires")
plt.xlabel("Prix (euros)")
plt.show()

Illisible : une seule barre collée à gauche. En cause, le prix maximum à
4 161 €, qui étire tout l'axe.

**C'est une information, pas un problème.** Elle confirme ce qu'on avait vu
en séance 2.1 (moyenne 3,93 € contre médiane 1,95 €). Zoomons sur la zone
utile :

In [ ]:
# 85,8 % des ventes sont a moins de 5 euros
ventes.query("prix < 10")["prix"].plot(kind="hist", bins=40, figsize=(7, 4))

# Le zoom se dit DANS le titre : sinon on cache une information au lecteur
plt.title("Repartition des prix unitaires (moins de 10 euros)")
plt.xlabel("Prix (euros)")
plt.show()

Voilà l'entreprise réelle : **un vendeur de petits articles à moins de 5 €**,
en gros volumes. Ce n'est pas ce qu'une moyenne de 3,93 € laissait deviner —
elle aurait pu décrire aussi bien un catalogue homogène autour de 4 €.

> Quand vous zoomez pour rendre un graphique lisible, **dites-le dans le
> titre**. « moins de 10 euros » dans le titre ci-dessus : sans cette
> mention, vous cachez une information à votre lecteur.

---

### ✏️ À vous 3a — La répartition des quantités

> **Votre mission :**
> - Même technique, autre colonne : tracer l'histogramme des **quantités** commandées, en 50 classes.
> - Puis compter les lignes de plus de 100 unités → `nb_grosses`.

In [ ]:
complet["qte"].plot(kind="hist", bins=50, figsize=(7, 4))
plt.title("Repartition des quantites commandees")
plt.xlabel("Quantite")
plt.show()

# Meme forme que pour les prix : une masse a gauche, une longue queue
# a droite qui rend le graphique illisible
nb_grosses = len(complet.query("qte > 100"))
print(nb_grosses, "lignes de plus de 100 unites")

In [ ]:
verifier("3a - lignes de plus de 100 unites", nb_grosses == 560,
         "le type de graphique est hist ; le seuil de la question est 100")

---

### ✏️ À vous 3b — Zoomer, et le dire

> **Votre mission :**
> - L'histogramme précédent est illisible : la commande à 1 440 unités étire tout l'axe.
> - Le retracer sur les seules lignes de **moins de 25 unités**, et compter combien de lignes cela représente → `nb_petites`.
> - ⚠️ Le zoom doit apparaître **dans le titre** : sans ça, vous cachez une information à votre lecteur.
> - Quelle part du fichier ces lignes représentent-elles ?

In [ ]:
petites = complet.query("qte < 25")
nb_petites = len(petites)

petites["qte"].plot(kind="hist", bins=25, figsize=(7, 4))
# Le zoom se dit DANS le titre, comme pour les prix
plt.title("Repartition des quantites (moins de 25 unites)")
plt.xlabel("Quantite")
plt.show()

print(nb_petites, "lignes sur", len(complet),
      "soit", round(100 * nb_petites / len(complet), 1), "%")

# 90,9 % des lignes portent sur moins de 25 unites. La mediane est a 8 :
# l'activite reelle, ce sont de petites commandes tres nombreuses.

In [ ]:
verifier("3b - lignes de moins de 25 unites", nb_petites == 41033,
         "filtrez avec query avant de tracer, puis len() sur le resultat")

## 4. Le nuage de points — chercher un lien

In [ ]:
# random_state=42 : le meme echantillon a chaque execution
echantillon = ventes.query("prix < 20 and qte < 200").sample(2000, random_state=42)

# alpha=0.3 : des points translucides, sinon 2 000 points font une tache
echantillon.plot(kind="scatter", x="prix", y="qte", alpha=0.3, figsize=(7, 4))
plt.title("Quantite commandee selon le prix unitaire")
plt.xlabel("Prix unitaire (euros)")
plt.ylabel("Quantite")
plt.show()

La relation est nette : **plus le prix unitaire est élevé, plus les quantités
commandées sont faibles.** Attendu, mais utile à vérifier.

> `alpha=0.3` rend les points semi-transparents : là où ils se superposent,
> la couleur devient plus dense. Sans ça, 2 000 points forment une bouillie
> noire.

> ⚠️ **Une relation n'est pas une cause.** Le prix ne « fait » pas baisser
> les quantités : ce sont deux conséquences du type de produit. Nous
> reviendrons sur cette distinction au bloc 5 (A/B testing) — c'est tout
> l'objet de l'expérimentation.

Ce nuage-ci ne montre que 2 000 lignes filtrées. À vous de le refaire sur le
fichier entier, et surtout de **mesurer** ce que l'œil croit voir.

---

### ✏️ À vous 4a — Mesurer le lien entre quantité et prix

> **Votre mission :**
> - Tracer un **nuage de points** avec `qte` en abscisse et `prix` en ordonnée.
> - `alpha=0.2` rend les points translucides : sans lui, 45 000 points forment une tache noire.
> - Mettre la corrélation entre les deux dans `lien`, arrondie à 3 décimales.

In [ ]:
# alpha=0.2 : sans transparence, 45 000 points font une tache noire
complet.plot(kind="scatter", x="qte", y="prix", alpha=0.2, figsize=(7, 4))
plt.title("Quantite commandee et prix unitaire")
plt.show()

# -0,024 : autant dire aucun lien. On commande beaucoup d'articles
# chers comme d'articles bon marche.
lien = round(complet["qte"].corr(complet["prix"]), 3)   ## entre -1 et +1
print(lien)

In [ ]:
verifier("4a - lien quantite / prix", lien == -0.024,
         "le type de graphique est scatter ; corr() donne le coefficient")

---

### ✏️ À vous 4b — Choisir le bon graphique

> **Votre mission :**
> - À chaque question sa figure. Compléter la liste `reponses` avec les quatre types, **dans l'ordre des questions** :
> - 1. Comment le chiffre d'affaires évolue-t-il dans le temps ?
> - 2. Quel pays est le plus gros marché ?
> - 3. Comment les prix sont-ils répartis ?
> - 4. Les grosses quantités vont-elles avec les prix bas ?

In [ ]:
# evolution -> courbe | classement -> barres | repartition -> histogramme
# | relation entre deux grandeurs -> nuage de points
reponses = ["line", "barh", "hist", "scatter"]

print(reponses)

In [ ]:
verifier("4b - le bon graphique pour la bonne question",
         reponses == ["line", "barh", "hist", "scatter"],
         "une evolution, un classement, une repartition, une relation")

## 5. Un graphique qu'on peut envoyer à un dirigeant

Les quatre lignes qui séparent un brouillon d'un livrable :

In [ ]:
ca_cat = complet.groupby("categorie")["ca"].sum().sort_values()

ca_cat.plot(kind="barh", figsize=(7, 4), color="#4C72B0")
plt.title("Chiffre d'affaires par categorie de produit, 2011")   ## quoi + quand
plt.xlabel("Chiffre d'affaires (euros)")   ## la grandeur et son unite
plt.ylabel("")            ## "categorie" n'apprend rien : on l'enleve
plt.tight_layout()        ## rien ne sera coupe au bord de la figure
plt.show()

- **`title`** : ce que montre le graphique, **et sur quelle période**
- **`xlabel` / `ylabel`** : le nom de la grandeur **et son unité**
- **`ylabel("")`** : on enlève l'étiquette « categorie », qui est évidente
- **`tight_layout()`** : évite que les étiquettes soient coupées

Si votre lecteur doit vous demander « c'est en quoi ? » ou « sur quelle
période ? », le graphique a échoué.

---

## Et maintenant : l'étude de cas

Vous avez tous les outils. Passez au notebook **« Étude de cas »**.

> 👥 **En binôme.** Un tient le clavier, l'autre lit l'énoncé et vérifie —
> puis vous échangez à mi-parcours. C'est la façon dont on travaille
> réellement sur une analyse, et c'est plus efficace que chacun de son côté.

---

# Corrigé de la feuille

Les exercices de la séance sont corrigés plus haut, dans le fil du cours. Les cellules ci-dessous rejouent le setup pour rester exécutables isolément.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")
clients = pd.read_csv(BASE + "clients.csv")
produits = pd.read_csv(BASE + "produits.csv")

ventes["ca"] = ventes["qte"] * ventes["prix"]     ## le CA de chaque ligne
ventes["date"] = pd.to_datetime(ventes["date"])   ## du texte vers des dates

# Deux jointures enchainees : ventes + clients, puis + produits
complet = ventes.merge(clients, on="client_id").merge(produits, on="prod_id")
print(complet.shape)   ## 45 123 lignes : aucune perdue en chemin

### Le contexte

> **Votre mission :**
> Votre direction prépare le budget de l'an prochain. Elle vous demande une note d'une page : **où est le chiffre d'affaires, et où sont les risques ?** Les sept étapes ci-dessous vous y mènent. La dernière est le livrable.

In [ ]:
print("Donnees chargees :", complet.shape[0], "lignes")

In [ ]:
verifier("0 - donnees pretes", len(complet) == 45123, "relancez la cellule de preparation")

### Étape 1 — La saisonnalité

> **Votre mission :**
> - Calculer le CA par mois dans `ca_mois` (index = le mois en texte, ex. `"2011-10"`).
> - Tracer une **courbe**, avec titre et libellé d'axe.
> - Mettre le meilleur mois dans `mois_top`.

In [ ]:
ca_mois = complet.groupby(complet["date"].dt.to_period("M"))["ca"].sum()
ca_mois.index = ca_mois.index.astype(str)   ## des etiquettes lisibles

ca_mois.plot(kind="line", marker="o", figsize=(7, 4))
plt.title("Nombre de commandes par mois")
plt.ylabel("commandes")
plt.xticks(rotation=45)
plt.show()

mois_top = ca_mois.idxmax()
print(mois_top)

In [ ]:
verifier("1 - meilleur mois", mois_top == "2011-10",
         "groupby sur le mois puis sum() sur ca")

### Étape 2 — Le piège de décembre

> **Votre mission :**
> - Le graphique montre une chute en décembre. **Avant de conclure**, comptez les jours de décembre présents dans les données → `jours_dec`.
> - Puis répondez : la chute est-elle réelle ? Mettez `True` ou `False` dans `chute_reelle`.

In [ ]:
dec = complet.query("date >= '2011-12-01'")
jours_dec = dec["date"].dt.day.nunique()   ## des JOURS distincts, pas des lignes

# 8 jours de vente compares a des mois complets : la comparaison
# n'a aucun sens. Il n'y a pas de chute, il y a un mois tronque.
chute_reelle = False

print(jours_dec, "jours de decembre |", "chute reelle :", chute_reelle)

In [ ]:
verifier("2a - jours de decembre", jours_dec == 8, "nunique() sur .dt.day")
verifier("2b - interpretation", chute_reelle is False,
         "8 jours face a des mois de 30 : les periodes ne sont pas comparables")

### Étape 3 — Les marchés

> **Votre mission :**
> - CA par pays, top 8, en **barres horizontales** triées.
> - Mettre le deuxième marché dans `marche_2`.

In [ ]:
ca_pays = complet.groupby("pays")["ca"].sum().nlargest(8)

# barh = barres horizontales : les noms de pays se lisent sans rotation.
# sort_values() croissant car matplotlib dessine de bas en haut.
ca_pays.sort_values().plot(kind="barh", figsize=(7, 4))
plt.title("Chiffre d'affaires par pays (top 8)")
plt.xlabel("CA (euros)")
plt.show()

# nlargest est deja trie : position 0 = 1er marche, position 1 = 2e
marche_2 = ca_pays.index[1]
print(marche_2)

In [ ]:
verifier("3 - deuxieme marche", marche_2 == "Irlande",
         "nlargest trie deja : l'index 1 est le deuxieme")

### Étape 4 — La concentration client

> **Votre mission :**
> - Calculer le CA par client, puis la part des **10 premiers** dans le CA total → `part_top10` (en %, arrondi à 1 décimale).
> - Compter les clients irlandais → `nb_irl`.
> - Ces deux chiffres sont le cœur de votre note.

In [ ]:
ca_cli = complet.groupby("client_id")["ca"].sum().sort_values(ascending=False)

part_top10 = round(100 * ca_cli.head(10).sum() / ca_cli.sum(), 1)   ## en %
nb_irl = complet.query("pays == 'Irlande'")["client_id"].nunique()   ## deux !

print(part_top10, "% du CA pour 10 clients |", nb_irl, "clients irlandais")

# 10 clients sur 472 font plus du tiers du chiffre d'affaires,
# et le deuxieme marche du groupe repose sur DEUX comptes.

In [ ]:
verifier("4a - part des 10 premiers", part_top10 == 37.5,
         "divisez la somme des 10 premiers par le total ca_cli.sum()")
verifier("4b - clients irlandais", nb_irl == 2, "nunique() sur client_id")

### Étape 5 — Le top produits, et ce qu'il révèle

> **Votre mission :**
> - Afficher les 5 produits qui génèrent le plus de CA → `top_prod`.
> - **Regardez les noms attentivement.** Deux d'entre eux ne sont pas des produits.
> - Mettre leurs deux libellés dans la liste `faux_produits`.

In [ ]:
top_prod = complet.groupby("libelle")["ca"].sum().nlargest(5).round(2)   ## top 5
print(top_prod)

# "Postage" = les frais de port. "Manual" = une saisie manuelle au comptoir.
# Ce sont des ecritures comptables, pas des articles du catalogue.
# Les laisser dans un classement produits fausse toute decision d'assortiment.
faux_produits = ["Postage", "Manual"]

In [ ]:
verifier("5 - faux produits reperes", sorted(faux_produits) == ["Manual", "Postage"],
         "un classement produits ne devrait pas contenir de frais de port")

### Étape 6 — Le classement corrigé

> **Votre mission :**
> - Refaire le top 5 en excluant `Postage` et `Manual` → `top_reel`.
> - Calculer la part de ces deux lignes dans le CA total → `part_faux` (en %, arrondi à 1 décimale).

In [ ]:
# @faux_produits : query() va chercher la variable Python definie plus haut
reels = complet.query("libelle not in @faux_produits")   ## "not in" : l'inverse
top_reel = reels.groupby("libelle")["ca"].sum().nlargest(5).round(2)
print(top_reel)

ca_faux = complet.query("libelle in @faux_produits")["ca"].sum()
part_faux = round(100 * ca_faux / complet["ca"].sum(), 1)
print(part_faux, "% du CA")

In [ ]:
verifier("6a - vrai produit leader", top_reel.index[0] == "Regency Cakestand 3 Tier",
         "filtrez avec 'not in @faux_produits' avant de classer")
verifier("6b - part des faux produits", part_faux == 6.2,
         "utilisez 'in @faux_produits' pour isoler ces deux lignes")

### Étape 7 — Le livrable

> **Votre mission :**
> - Rédigez votre note dans la cellule markdown ci-dessous, en remplaçant les points de suspension.
> - **Trois recommandations, chacune appuyée sur un chiffre que vous avez calculé.**
> - Un constat n'est pas une recommandation : « l'Irlande fait 22,7 % du CA » est un constat ; « il faut sécuriser ces deux contrats » est une recommandation.
> - 👥 Chaque binôme présentera 3 minutes.

In [ ]:
print("meilleur mois        :", mois_top)
print("2e marche            :", marche_2, "avec", nb_irl, "clients")
print("part des 10 premiers :", part_top10, "%")
print("faux produits        :", part_faux, "% du CA")

# ---------------------------------------------------------------------
# Note attendue (une redaction parmi d'autres) :
#
# 1. RISQUE DE CONCENTRATION — 10 clients sur 472 pesent 37,5 % du chiffre
#    d'affaires, et le 2e marche du groupe (l'Irlande, 22,7 % du CA) repose
#    sur DEUX comptes. Recommandation : securiser ces contrats par des
#    engagements pluriannuels, et ne pas traiter l'Irlande comme un marche
#    a developper mais comme une dependance a couvrir.
#
# 2. SAISONNALITE — le pic est en octobre, pas en decembre : nos clients
#    sont des detaillants qui se reapprovisionnent AVANT Noel.
#    Recommandation : avancer les operations commerciales de six semaines
#    par rapport au calendrier grand public.
#    (Et la "chute" de decembre est un artefact : le fichier s'arrete au 9.)
#
# 3. QUALITE DES DONNEES — 6,2 % du CA est porte par "Postage" et "Manual",
#    qui ne sont pas des produits. Recommandation : les sortir du perimetre
#    avant toute decision d'assortiment, sans quoi les frais de port
#    apparaissent comme notre meilleure vente.
# ---------------------------------------------------------------------

In [ ]:
verifier("7 - tous les chiffres disponibles",
         all(v is not None for v in [mois_top, marche_2, part_top10, part_faux]),
         "reprenez les etapes 1 a 6 avant de rediger")
print()
print("A vous : ajoutez une cellule de texte et redigez vos 3 recommandations.")

---

## Ce que vous savez faire maintenant

| Votre question | Le graphique | La commande |
|---|---|---|
| comment ça évolue ? | courbe | `serie.plot(kind="line")` |
| qui est le plus gros ? | barres | `serie.plot(kind="barh")` |
| comment c'est réparti ? | histogramme | `df["prix"].plot(kind="hist", bins=30)` |
| y a-t-il un lien ? | nuage de points | `df.plot(kind="scatter", x="qte", y="prix")` |

Et toujours :

```python
plt.title("Ce que montre le graphique")
plt.xlabel("Nom de l'axe (unite)")
plt.ylabel("Nom de l'axe (unite)")
plt.show()
```

## Les trois réflexes qui font la différence

1. **Un graphique sans titre ni unité n'est pas un livrable.** Si votre
   lecteur doit vous demander « c'est en quoi ? », vous avez raté.
2. **Regardez toujours ce que le graphique ne montre pas.** Un mois incomplet,
   une catégorie absente, un axe qui ne part pas de zéro.
3. **Un chiffre ne devient une recommandation que quand il est rattaché à une
   décision.** « L'Irlande fait 22,7 % du CA » est un constat. « 22,7 % du CA
   repose sur deux comptes, il faut sécuriser ces contrats » est une
   recommandation.